<a href="https://colab.research.google.com/github/AmitoVrito/Traceprop/blob/main/notebooks/runs/review_runs_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traceprop review runs (items 1, 2, 8)

Produces the result files `docs/mlsys/REVIEW_SCAFFOLD.md` needs. Run on an **A100** runtime;
job 4 is CPU-only and runs fine in the same A100 session. **Item 9 (rank agreement) is excluded**:
it needs a score-matrix step not yet wired.

Each run writes to `experiments/results/`; the last cell copies them to Drive. Download these back
into your local `experiments/results/`, then tell the assistant:

- `exp36_memory_storage_pythia1b.json`
- `exp35_inlineprecond_overhead_hf_pythia7b_split.json`
- `exp35_overhead_1b_k8.json`, `exp35_overhead_1b_k32.json`, `exp35_overhead_1b_k64.json`
- `exp35_ksweep_seed0.npz` ... `exp35_ksweep_seed4.npz`

`exp36` is a new script not yet run on GPU; if it errors, send the traceback.

In [1]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
print('GPU:', name)
if 'A100' not in name:
    print('WARNING: jobs 1-3 expect an A100 (40GB); results on a different GPU are not comparable '
          'to the paper. Job 4 (quality sweep) is CPU-only and is fine on any runtime.')

# Install matches the working stage2 notebook: use Colab's current numpy/transformers/peft,
# drop torchao (its version check breaks peft LoRA dispatch), install logix-ai for the LogIX arm.
!pip -q install datasets scipy scikit-learn accelerate
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install --ignore-requires-python logix-ai
import numpy, transformers, peft
print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)


Python 3.13.15
NVIDIA A100-SXM4-40GB, 40960 MiB
GPU: NVIDIA A100-SXM4-40GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.3 MB/s eta 0:00:00
numpy 2.1.3 | transformers 5.20.1 | peft 0.21.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)


Mounted at /content/drive


In [3]:
# Repo is public; press Enter at the prompt for an anonymous clone, or paste a token.
import getpass
TOKEN = getpass.getpass('GitHub token (Enter for public clone): ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git' if TOKEN else 'https://github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!git log --oneline -1
!pip -q install -e .
%cd /content/Traceprop/experiments


GitHub token (Enter for public clone): ··········
/content/Traceprop
e7e8a34 (HEAD -> main, origin/main, origin/HEAD) Fix Job 5 cell: '%cd ... &&' doesn't chain; use '!cd ... && git pull' then %cd experiments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments


In [4]:
!mkdir -p results
!cp /content/drive/MyDrive/traceprop_runs/*.json results/ 2>/dev/null
!cp /content/drive/MyDrive/traceprop_runs/*.npz results/ 2>/dev/null
!ls results/


exp25b_overhead_vs_modelsize.json
exp25_hf_EleutherAI_pythia-1b_track1.json
exp25_hf_EleutherAI_pythia-2.8b_track1.json
exp25_hf_EleutherAI_pythia-410m_track1.json
exp25_hf_gpt2_track1.json
exp25_hf_pythia-1b_track1.json
exp25_hf_pythia-2.8b_track1.json
exp25_hf_pythia-410m_track1.json
exp25_pythia1b_track0_factored_kfac8.json
exp25_pythia1b_track1_factored_kfac8.json
exp25_pythia1b_track1_matched256.json
exp25_pythia1b_track6_factored_kfac8.json
exp25_pythia1b_track6_matched1536.json
exp25_tiny_tiny-gpt.json
exp25_tiny_tiny-gpt_track1.json
exp26_headtohead_repeats20_summary.json
exp26_hf_gpt2_track1.json
exp26_tiny_tiny-gpt_track1.json
exp27_hf_gpt2.json
exp27_tiny_tiny-clf.json
exp28_hf_gpt2.json
exp28_tiny_frozen-synth.json
exp29_hf_gpt2.json
exp29_hf_gpt2_rerun.json
exp29_hf_gpt2_rerun_raw.npz
exp29_tiny_tiny-clf.json
exp30_same_setting_overhead.json
exp31_logix_comparison.json
exp31_logix_hf_gpt2.json
exp31_logix_tiny_tiny-clf.json
exp31_pythia1b_track0_fixed2.json
exp31_pythia1b_

## Job 1 — memory/storage (item 1). A100, ~5 min.

In [ ]:
!python exp36_memory_storage.py --model EleutherAI/pythia-1b --device cuda \
    --steps 200 --batch 8 --seq 128 --kfac 8 --track 0 \
    --out results/exp36_memory_storage_pythia1b.json


python3: can't open file '/content/exp36_memory_storage.py': [Errno 2] No such file or directory


## Job 2 — 6.9B logging/covariance split (item 2). A100, ~15-20 min.

Same as the saved 6.9B run but WITHOUT `--skip_log_only`, so the logging-only arm is timed.

In [ ]:
!python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-6.9b \
    --device cuda --dtype auto --track 0 --kfac 8 --steps 50 --batch 8 --seq 128 \
    --repeats 5 --with_logix --skip_logix_pca \
    --out results/exp35_inlineprecond_overhead_hf_pythia7b_split.json


python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory


## Job 3 — k-sweep overhead at 1B (item 8, overhead half). A100, ~30-40 min.

In [ ]:
for k in [8, 32, 64]:
    print(f'=== k={k} ===')
    !python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-1b \
        --device cuda --dtype fp32 --track 0 --kfac {k} --steps 200 --batch 8 \
        --seq 128 --repeats 10 --with_logix \
        --out results/exp35_overhead_1b_k{k}.json


=== k=8 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory
=== k=32 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory
=== k=64 ===
python3: can't open file '/content/exp35_inlineprecond_overhead_hf.py': [Errno 2] No such file or directory


## Job 4 — k-sweep quality (item 8, quality half). CPU-only; runs in the same session, ~15-25 min.

In [ ]:
for s in range(5):
    print(f'=== seed {s} ===')
    !python exp35_logix_lds.py --backend tiny --data synthetic --device cpu \
        --track 0 --factored --inline_precond --tune_logix --lora_init random \
        --kfac_values 8,32,64 --seed {s} \
        --out results/exp35_ksweep_seed{s}.npz


=== seed 0 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 1 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 2 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 3 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory
=== seed 4 ===
python3: can't open file '/content/exp35_logix_lds.py': [Errno 2] No such file or directory


## Save results to Drive

Then download these from `MyDrive/traceprop_runs/` into your local `experiments/results/`.

In [ ]:
!cp results/exp36_memory_storage_pythia1b.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_inlineprecond_overhead_hf_pythia7b_split.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_overhead_1b_k*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_ksweep_seed*.npz /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('copied to Drive:')
!ls -la /content/drive/MyDrive/traceprop_runs/ | grep -E 'exp36|pythia7b_split|overhead_1b_k|ksweep'


copied to Drive:
-rw-------  1 root root    1497 Oct 19 21:23 exp35_inlineprecond_overhead_hf_pythia7b_split.json
-rw-------  1 root root    1967 Oct 19 21:23 exp35_overhead_1b_k32.json
-rw-------  1 root root    1968 Oct 19 21:23 exp35_overhead_1b_k64.json
-rw-------  1 root root    1958 Oct 19 21:23 exp35_overhead_1b_k8.json
-rw-------  1 root root    1212 Oct 19 21:23 exp36_memory_storage_pythia1b.json


In [5]:
import os
if not os.path.isdir('/content/Traceprop/.git'):
    !git clone -q https://github.com/AmitoVrito/Traceprop.git /content/Traceprop
!cd /content/Traceprop && git pull -q && git log --oneline -1 && pip -q install -e . 2>/dev/null
%cd /content/Traceprop/experiments
!python exp35_logix_lds.py --backend hf --model EleutherAI/pythia-1b --data synthetic \
    --device cuda --skip_gpu_check \
    --factored --inline_precond --lora_init random --tune_logix --logix_permodule_damping \
    --n_train 1600 --n_test 128 --epochs 2 \
    --precond_damping_grid 1e-4,1e-3,1e-2 --rank_deltas 1e-3,1e-4,1e-2 \
    --rank_agreement_out results/exp39_rank_agreement_1b.npz
!cp results/exp39_rank_agreement_1b_report.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('done; report copied to Drive if it exists')

e7e8a34 (HEAD -> main, origin/main, origin/HEAD) Fix Job 5 cell: '%cd ... &&' doesn't chain; use '!cd ... && git pull' then %cd experiments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments
[exp35] training target model (1600 examples) ...
config.json: 100% 569/569 [00:00<00:00, 2.19MB/s]

model.safetensors: downloading bytes:   0% 0.00/2.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  13% 263M/2.09G [00:01<00:05, 323MB/s, 18.6MB/s  ]
model.safetensors: reconstructing file:   3% 67.1M/2.09G [00:02<00:48, 41.9MB/s, 1.39kB/s  ]
model.safetensors: downloading bytes:  26% 537M/2.09G [00:02<00:02, 573MB/s, 40.9MB/s  ]
model.safetensors: downloading bytes:  50% 1.05G/2.09G [00:03<00:01, 641MB/s

In [6]:
import os
if not os.path.isdir('/content/Traceprop/.git'):
    !git clone -q https://github.com/AmitoVrito/Traceprop.git /content/Traceprop
!cd /content/Traceprop && git pull -q && git log --oneline -1 && pip -q install -e . 2>/dev/null
%cd /content/Traceprop/experiments
!python exp35_logix_lds.py --backend hf --model EleutherAI/pythia-1b --data synthetic \
    --device cuda --skip_gpu_check --track 0 \
    --factored --inline_precond --lora_init random --tune_logix --logix_permodule_damping \
    --n_train 1600 --n_test 128 --epochs 2 \
    --precond_damping_grid 1e-4,1e-3,1e-2 --rank_deltas 1e-3,1e-4,1e-2 \
    --rank_agreement_out results/exp39_rank_agreement_1b.npz
!cp results/exp39_rank_agreement_1b_report.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('done; report copied to Drive if it exists')


b46469d (HEAD -> main, origin/main, origin/HEAD) Job 5: add --track 0 (all-layer, 34 modules) to match the headline scope; was defaulting to --track 1 (last block)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments
[exp35] training target model (1600 examples) ...
Loading weights: 100% 195/195 [00:00<00:00, 5883.33it/s]
[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-1b
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized 

In [9]:
%cd /content/Traceprop/experiments
import sys
sys.path.insert(0, "/content/Traceprop")     # make 'traceprop' importable in-kernel
import importlib; importlib.invalidate_caches()
import torch, torch.nn as nn, transformers, peft
from exp27_lds_quality import build_hf_classifier
from traceprop.llm import select_lora_linears

m = build_hf_classifier("EleutherAI/pythia-1b", r=8)
A = [n for n, _ in select_lora_linears(m, ("lora_A","lora_B","score","classifier"), last_n_blocks=None)]
B = [n for n, mod in m.named_modules() if isinstance(mod, nn.Linear) and ("lora_A" in n or "lora_B" in n)]

print("stack: transformers", transformers.__version__, "| peft", peft.__version__)
print("A overhead select_lora_linears:", len(A))
print("B exp35 LogIX tracked_names   :", len(B))
print("in A not in B:", sorted(set(A) - set(B)))
print("in B not in A:", sorted(set(B) - set(A)))

/content/Traceprop/experiments


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-1b
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


stack: transformers 5.20.1 | peft 0.21.2
A overhead select_lora_linears: 34
B exp35 LogIX tracked_names   : 32
in A not in B: ['base_model.model.score.modules_to_save.default', 'base_model.model.score.original_module']
in B not in A: []


In [10]:
!cd /content/Traceprop && git pull -q && git log --oneline -1 && pip -q install -e .
%cd /content/Traceprop/experiments
!python exp36_memory_storage.py --model EleutherAI/pythia-1b --device cuda \
    --steps 200 --batch 8 --seq 128 --kfac 8 --track 0 \
    --out results/exp36_memory_storage_pythia1b.json
!cp results/exp36_memory_storage_pythia1b.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('done; exp36 json copied to Drive if it exists')

1469067 (HEAD -> main, origin/main, origin/HEAD) exp36: per-module sketch-entry counts + 34-count assert; logger tracks per-module sketch blocks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments
Loading weights: 100% 195/195 [00:00<00:00, 4446.45it/s]
[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-1b
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downst